In [422]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)

channels = ["Email", "Social", "PPC", "SEO", "Display"]
campaigns = ["Spring Sale", "Winter Promo", "New Year Blast", "Product Launch"]
date_formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S"]
base_date = datetime(2024, 1, 1, 0, 0, 0)

data = []
for i in range(100):
    touchpoint_id = 1001 + i
    customer_id = f"CUST{random.randint(100, 119)}" if random.random() > 0.05 else ""
    if random.random() < 0.1:
        customer_id = customer_id.replace("T", "Y") if customer_id else ""
    
    # Fix: Safely extract number or default to 0
    try:
        cust_num = int(customer_id.replace("CUST", "").replace("Y", "").replace("T", "")) if customer_id else 0
    except ValueError:
        cust_num = 0
    channel_weights = [0.4, 0.2, 0.2, 0.1, 0.1] if cust_num % 2 == 0 else [0.1, 0.3, 0.3, 0.2, 0.1]
    channel = random.choices(channels, weights=channel_weights, k=1)[0]
    if random.random() < 0.15:
        channel = channel.lower() if random.random() < 0.5 else {"PPC": "PayPerClick", "SEO": "Search"}.get(channel, channel)
    
    days_to_add = random.randint(0, 4)
    hours = random.randint(0, 23)
    minutes = random.randint(0, 59)
    seconds = random.randint(0, 59)
    interaction_date = base_date + timedelta(days=sum(random.randint(0, 4) for _ in range(i)), 
                                             hours=hours, minutes=minutes, seconds=seconds)
    date_format = random.choice(date_formats)
    if random.random() < 0.03:
        interaction_date = "2024-02-30 10:00:00"
    else:
        interaction_date = interaction_date.strftime(date_format)
    
    campaign_name = random.choice(campaigns) if random.random() > 0.1 else ""
    if random.random() < 0.1:
        campaign_name = campaign_name + " 2024" if campaign_name else ""
    
    cost = random.uniform(1, 50)
    if random.random() < 0.05:
        cost = random.choice(["five", "ten", "twenty"])
    elif random.random() < 0.03:
        cost = random.choice([9999, -10])
    
    conversion = random.choice(["Yes", "No"]) if random.random() > 0.07 else ""
    if random.random() < 0.05:
        conversion = random.choice(["Yess", "maybe", "NOO"])

    data.append([touchpoint_id, customer_id, channel, interaction_date, campaign_name, cost, conversion])

df = pd.DataFrame(data, columns=["TouchpointID", "CustomerID", "Channel", 
                                 "InteractionDate", "CampaignName", "Cost", "Conversion"])
df.to_csv("messy_marketing_attribution_with_dups.csv", index=False)
print("Dataset saved as 'messy_marketing_attribution_with_dups.csv'!")

Dataset saved as 'messy_marketing_attribution_with_dups.csv'!


In [423]:
df_clean = df.copy()

In [424]:
df_clean.head()

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion
0,1001,CUST110,Email,01/01/2024 15:29:28,New Year Blast,10.754662,Yess
1,1002,CUST107,Social,2024-01-04 18:41:52,Winter Promo,32.473109,NOO
2,1003,CUST103,email,01/07/2024 00:59:26,,26.769088,No
3,1004,CUST112,Email,2024-01-03 03:09:31,Winter Promo,23.441593,No
4,1005,CUSY101,PPC,2024-01-11 20:33:24,Winter Promo,13.358611,No


In [425]:
# get the number of empty values in the column IntercationDate
df_clean['InteractionDate'][df_clean['InteractionDate'] == ""].sum()

0

In [426]:
df_clean['InteractionDate'] = pd.to_datetime(df_clean['InteractionDate'], errors='coerce')
df_clean.sort_values('InteractionDate').reset_index(drop=True)

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion
0,1001,CUST110,Email,2024-01-01 15:29:28,New Year Blast,10.754662,Yess
1,1003,CUST103,email,2024-01-07 00:59:26,,26.769088,No
2,1007,CUST115,Social,2024-01-12 17:16:23,New Year Blast,38.091852,Yes
3,1018,CUST116,Email,2024-01-29 09:43:25,Product Launch,37.42422,Yes
4,1027,CUST114,Email,2024-02-16 03:25:26,Product Launch,7.16626,
...,...,...,...,...,...,...,...
95,1095,CUST110,social,NaT,Spring Sale,48.044559,No
96,1096,CUST105,SEO,NaT,Spring Sale,11.065702,No
97,1098,CUST114,email,NaT,New Year Blast,34.321752,No
98,1099,CUST118,PPC,NaT,Spring Sale,3.460768,Yes


In [427]:
df_clean['InteractionDate'] = df_clean['InteractionDate'].ffill() 
df_clean.head()

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion
0,1001,CUST110,Email,2024-01-01 15:29:28,New Year Blast,10.754662,Yess
1,1002,CUST107,Social,2024-01-01 15:29:28,Winter Promo,32.473109,NOO
2,1003,CUST103,email,2024-01-07 00:59:26,,26.769088,No
3,1004,CUST112,Email,2024-01-07 00:59:26,Winter Promo,23.441593,No
4,1005,CUSY101,PPC,2024-01-07 00:59:26,Winter Promo,13.358611,No


In [428]:
channel_map = {
    'email': 'Email', 'Email': 'Email',
    'social': 'Social', 'Social': 'Social',
    'PPC': 'PPC', 'PayPerClick': 'PPC',
    'SEO': 'SEO', 'Search': 'SEO',
    'Display': 'Display', 'display': 'Display',
    'ppc': 'PPC',
    'seo': 'SEO'
}
df_clean['Channel'] = df_clean['Channel'].replace(channel_map)
df_clean['Channel'].value_counts()

Channel
PPC        33
Email      25
Social     18
SEO        18
Display     6
Name: count, dtype: int64

In [429]:
df_clean['Cost'][df_clean['Cost'].str.contains(r'\D', regex=True, na=False)]

18    five
22     ten
30     ten
Name: Cost, dtype: object

In [430]:
cost_map = {
    'five': 5,
    'twenty': 20,
    'ten': 10
}

df_clean['Cost'] = pd.to_numeric(df_clean['Cost'].replace(cost_map)).round(2)
df_clean
df_clean['Cost'].value_counts()

/var/folders/1x/_bv_nkxs7m11_bfrc7gxn84w0000gn/T/ipykernel_5271/1837830867.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean['Cost'] = pd.to_numeric(df_clean['Cost'].replace(cost_map)).round(2)


Cost
13.76    2
10.00    2
10.75    1
36.29    1
30.97    1
        ..
23.73    1
37.83    1
24.38    1
33.01    1
27.92    1
Name: count, Length: 98, dtype: int64

In [431]:
channels_median = df_clean.groupby('Channel')['Cost'].median()
channels_median

Channel
Display    29.595
Email      26.770
PPC        21.020
SEO        23.495
Social     30.195
Name: Cost, dtype: float64

In [432]:
def replace_outlier(row):
    if row['Cost'] < 0 or row['Cost'] > 100:
        channel = row['Channel']
        median = channels_median[channel]
        return median
    return row['Cost']

#testing the function replace_outlier
test = df_clean['Cost']
test = df_clean.apply(replace_outlier, axis=1)
test.value_counts().sort_index(ascending=False)

49.66    1
49.41    1
48.04    1
47.30    1
46.81    1
        ..
4.88     1
3.79     1
3.46     1
2.33     1
1.42     1
Name: count, Length: 98, dtype: int64

In [433]:
df_clean['Cost'] = df_clean.apply(replace_outlier, axis=1)
df_clean.describe()

,TouchpointID,InteractionDate,Cost
count,100.000000,100,100.000000
mean,1050.500000,2024-04-01 10:35:29.339999488,24.781100
min,1001.000000,2024-01-01 15:29:28,1.420000
25%,1025.750000,2024-02-11 16:59:55.750000128,11.415000
50%,1050.500000,2024-04-02 23:01:17,24.335000
75%,1075.250000,2024-05-22 04:45:07,36.805000
max,1100.000000,2024-07-15 05:58:54,49.660000
std,29.011492,NaN,14.152647


In [434]:
df_clean['Conversion'].value_counts()

Conversion
No       44
Yes      44
          5
NOO       3
Yess      2
maybe     2
Name: count, dtype: int64

In [435]:
conversion_map = {
    'Yes': 'Yes',
    'Yess': 'Yes',
    'No': 'No',
    '': 'No',
    'maybe': 'No',
    'NOO': 'No'
}

df_clean['Conversion'] = df_clean['Conversion'].replace(conversion_map)
df_clean['Conversion'].value_counts()
df_clean

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion
0,1001,CUST110,Email,2024-01-01 15:29:28,New Year Blast,10.75,Yes
1,1002,CUST107,Social,2024-01-01 15:29:28,Winter Promo,32.47,No
2,1003,CUST103,Email,2024-01-07 00:59:26,,26.77,No
3,1004,CUST112,Email,2024-01-07 00:59:26,Winter Promo,23.44,No
4,1005,CUSY101,PPC,2024-01-07 00:59:26,Winter Promo,13.36,No
...,...,...,...,...,...,...,...
95,1096,CUST105,SEO,2024-06-20 14:50:28,Spring Sale,11.07,No
96,1097,CUST110,SEO,2024-07-15 05:58:54,Spring Sale,31.64,Yes
97,1098,CUST114,Email,2024-07-15 05:58:54,New Year Blast,34.32,No
98,1099,CUST118,PPC,2024-07-15 05:58:54,Spring Sale,3.46,Yes


In [436]:
df_clean.duplicated(subset=['CustomerID', 'Channel']).value_counts()

False    67
True     33
Name: count, dtype: int64

In [437]:
df_clean = df_clean.loc[df_clean['CustomerID'] != ""].reset_index(drop=True)
df_clean

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion
0,1001,CUST110,Email,2024-01-01 15:29:28,New Year Blast,10.75,Yes
1,1002,CUST107,Social,2024-01-01 15:29:28,Winter Promo,32.47,No
2,1003,CUST103,Email,2024-01-07 00:59:26,,26.77,No
3,1004,CUST112,Email,2024-01-07 00:59:26,Winter Promo,23.44,No
4,1005,CUSY101,PPC,2024-01-07 00:59:26,Winter Promo,13.36,No
...,...,...,...,...,...,...,...
91,1096,CUST105,SEO,2024-06-20 14:50:28,Spring Sale,11.07,No
92,1097,CUST110,SEO,2024-07-15 05:58:54,Spring Sale,31.64,Yes
93,1098,CUST114,Email,2024-07-15 05:58:54,New Year Blast,34.32,No
94,1099,CUST118,PPC,2024-07-15 05:58:54,Spring Sale,3.46,Yes


In [438]:
df_clean = df_clean.sort_values(['CustomerID', 'Channel', 'InteractionDate']).reset_index(drop=True)

df_clean['Date'] = pd.to_datetime(df_clean['InteractionDate'], errors='coerce')
time_diff = df_clean['Date'].diff().abs() <= pd.Timedelta(minutes=5)
time_diff = time_diff.fillna(False)

near_dups = df_clean.duplicated(subset=['CustomerID', 'Channel'], keep=False) & time_diff
print(f'Near-duplicates (1hr): {near_dups.sum()}')

Near-duplicates (1hr): 4


In [439]:
df_clean = df_clean[~near_dups].reset_index(drop=True)
df_clean

,TouchpointID,CustomerID,Channel,InteractionDate,CampaignName,Cost,Conversion,Date
0,1036,CUST100,Email,2024-03-10 14:01:39,Spring Sale,49.41,No,2024-03-10 14:01:39
1,1041,CUST100,PPC,2024-03-28 03:11:05,New Year Blast,2.33,Yes,2024-03-28 03:11:05
2,1052,CUST101,Display,2024-04-04 05:14:22,Winter Promo,46.56,No,2024-04-04 05:14:22
3,1065,CUST101,PPC,2024-05-22 04:45:07,Product Launch,21.71,No,2024-05-22 04:45:07
4,1079,CUST101,PPC,2024-05-31 11:54:43,Spring Sale,40.90,No,2024-05-31 11:54:43
...,...,...,...,...,...,...,...,...
87,1063,CUSY103,PPC,2024-05-22 04:45:07,Spring Sale,24.29,No,2024-05-22 04:45:07
88,1069,CUSY108,SEO,2024-04-28 18:21:08,Spring Sale,8.02,No,2024-04-28 18:21:08
89,1045,CUSY111,Email,2024-03-19 13:17:25,Spring Sale,42.63,Yes,2024-03-19 13:17:25
90,1031,CUSY112,PPC,2024-02-18 15:22:36,Product Launch,10.00,Yes,2024-02-18 15:22:36
